In [21]:
from datasets import load_from_disk
from pprint import pprint
from transformers import AutoTokenizer

from arxiv_paper_discovery.config import PROCESSED_DATA_DIR
from arxiv_paper_discovery.label_taxonomy import labels_to_multihot, multihot_to_labels
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
arxiv_taxonomy_dataset = load_from_disk(PROCESSED_DATA_DIR / "arxiv_taxonomy_dataset")

print(arxiv_taxonomy_dataset)

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'abstract', 'categories', 'authors', 'update_date', 'labels'],
        num_rows: 2384152
    })
    val: Dataset({
        features: ['id', 'title', 'abstract', 'categories', 'authors', 'update_date', 'labels'],
        num_rows: 298019
    })
    test: Dataset({
        features: ['id', 'title', 'abstract', 'categories', 'authors', 'update_date', 'labels'],
        num_rows: 298019
    })
})


### Labels to multihot

In [37]:
import random

sample = arxiv_taxonomy_dataset["train"][random.randrange(len(arxiv_taxonomy_dataset["train"]))]

# Replace labels with multihot vector
pprint(sample, width=100)
sample["labels"] = labels_to_multihot(sample["labels"])
print(f"\nMultihot labels: {sample['labels']}")

# Optional sanity check: decode back
label_from_multihot = multihot_to_labels(sample["labels"])
print(f"\nLabel from multihot: {label_from_multihot}")

{'abstract': 'A fully nonlinear, time-asymptotic theory of resonant particle trapping in '
             'large-amplitude quasi-parallel Alfven waves is presented. The effect of trapped '
             'particles on the nonlinear dynamics of quasi-stationary Alfvenic discontinuities and '
             'coherent Alfven waves is highly non-trivial and forces to a significant departure of '
             'the theory from the conventional DNLS and KNLS equation models. The virial theorem '
             'is used to determine the time-asymptotic distribution function.',
 'authors': 'M.V. Medvedev, P.H. Diamond, M.N. Rosenbluth, V.I. Shevchenko (UCSD)',
 'categories': ['physics.plasm-ph', 'astro-ph', 'nlin.PS', 'patt-sol', 'physics.space-ph'],
 'id': 'physics/9803022',
 'labels': ['Applied and Interdisciplinary Physics',
            'Astrophysics',
            'Physics Other',
            'Nonlinear Dynamics'],
 'title': 'Asymptotic Theory of Particle Trapping in Coherent Nonlinear Alfven Waves'

### Tokenization

In [40]:
pprint(sample, width=100)

{'abstract': 'A fully nonlinear, time-asymptotic theory of resonant particle trapping in '
             'large-amplitude quasi-parallel Alfven waves is presented. The effect of trapped '
             'particles on the nonlinear dynamics of quasi-stationary Alfvenic discontinuities and '
             'coherent Alfven waves is highly non-trivial and forces to a significant departure of '
             'the theory from the conventional DNLS and KNLS equation models. The virial theorem '
             'is used to determine the time-asymptotic distribution function.',
 'authors': 'M.V. Medvedev, P.H. Diamond, M.N. Rosenbluth, V.I. Shevchenko (UCSD)',
 'categories': ['physics.plasm-ph', 'astro-ph', 'nlin.PS', 'patt-sol', 'physics.space-ph'],
 'id': 'physics/9803022',
 'labels': [1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
 'title': 'Asymptotic Theory of Particle Trapping in Coherent Nonlinear Alfven Waves',
 'update_date': datetime.datetime(2007, 5, 23, 0, 0)}


In [115]:
tokenizer = AutoTokenizer.from_pretrained("allenai/scibert_scivocab_uncased")
tokenized_sample = tokenizer(
    sample["title"],
    sample["abstract"],
    truncation=True, 
    padding=False, 
    max_length=512
    )

for k, v in tokenized_sample.items():
    print(f"{k}: {v}")

print(f"\nDecoded tokens: {tokenizer.decode(tokenized_sample['input_ids'])}")

KeyError: 'title'

In [116]:
### Tokenized Dataset Exploration
tokenized_dataset = load_from_disk(PROCESSED_DATA_DIR / "tok_scibert_scivocab_uncased")

print(tokenized_dataset)

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2384152
    })
    val: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 298019
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 298019
    })
})


In [117]:
tokenizer = AutoTokenizer.from_pretrained("allenai/scibert_scivocab_uncased")

tokenized_sample = tokenized_dataset["train"][random.randrange(len(tokenized_dataset["train"]))]

for k, v in tokenized_sample.items():
    print(f"{k}: {v}")

print(f"Token count: {len(tokenized_sample['input_ids'])}")
print(f"\nDecoded tokens: {tokenizer.decode(tokenized_sample['input_ids'])}")
print(f"Labels: {multihot_to_labels(tokenized_sample['labels'])}")

labels: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0]
input_ids: [102, 9153, 9473, 7392, 862, 11750, 579, 11853, 899, 131, 106, 2121, 1168, 579, 5076, 8785, 22988, 140, 2188, 103, 111, 626, 131, 10222, 121, 5676, 5877, 10311, 147, 1126, 422, 6920, 214, 111, 965, 147, 2035, 1127, 21789, 5212, 555, 188, 2193, 24115, 30113, 422, 7699, 3521, 422, 137, 1661, 579, 1953, 1865, 8515, 205, 121, 238, 4940, 422, 185, 26438, 168, 145, 137, 3768, 546, 106, 428, 11446, 1139, 603, 3925, 16826, 121, 111, 16727, 205, 428, 10699, 26975, 208, 19052, 111, 3972, 131, 5971, 849, 190, 22137, 11731, 137, 25287, 1852, 935, 1767, 422, 198, 15622, 1948, 988, 555, 188, 11750, 26731, 137, 4417, 27573, 205, 121, 111, 982, 422, 185, 3401, 130, 8513, 869, 579, 14335, 6097, 5264, 147, 5610, 111, 1411, 131, 8785, 10222, 121, 111, 2220, 131, 2385, 9428, 2393, 5445, 137, 5365, 3878, 205, 103]
token_type_ids: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1,